# Integrated Gradients (Captum)

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Integrated Gradients atribui a predição do modelo às entradas integrando gradientes ao longo de uma linha reta entre um baseline (e.g. zeros) e a entrada real. A atribuição soma exatamente a *diferença* da saída, satisfazendo o axioma de *completude*.


## Formulação Matemática

$$\text{IG}_i(x) = (x_i - x_i^\text{base}) \int_0^1 \frac{\partial f(x^\text{base} + \alpha(x - x^\text{base}))}{\partial x_i}\,d\alpha$$


## Implementação


In [ ]:
# pip install captum
import torch
import torch.nn as nn
from captum.attr import IntegratedGradients


In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 2))
    def forward(self, x): return self.net(x)

torch.manual_seed(0)
model = TinyClassifier().eval()


## Experimento


In [ ]:
x = torch.tensor([[0.5, -1.0, 2.0, 0.0]])
baseline = torch.zeros_like(x)
ig = IntegratedGradients(model)
attr = ig.attribute(x, baselines=baseline, target=1, n_steps=64)
print('attribution per feature:', attr.squeeze().tolist())
print('sum of attributions  :', attr.sum().item())
print('output diff          :', (model(x)[0,1] - model(baseline)[0,1]).item())


## Discussão

- A escolha do baseline importa: zero, média do dataset ou baseline ruidoso geram sinais diferentes.
- Use `n_steps=50`–`200`; mais passos reduzem o erro de discretização ao custo de mais compute.
- Captum integra com qualquer modelo PyTorch — visão, transformers (atribuição via camada de embedding), tabular.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
